# XRF55 Bench — WavMambaHAR

**Model:** WavMambaHAR baseline — XRF55 benchmark  
**Dataset:** XRF55 — 11 activities, 4620 train / 1980 test  
**Split:** train = reps 1–14, test = reps 15–20. No val.

---

## Attached dataset

| Dataset name | Mount path |
|---|---|
| `xrf55-amp-dataset` | `/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/` |

## Protocol reference

| # | Optimizer | LR | WD | Scheduler | Epochs | Notes |
|---|---|---|---|---|---|---|
| 01 | AdamW | 1e-4 | 0.01 | None | 40 | tf_mamba paper |
| 02 | Adam | 1e-3 | 0.0 | MultiStepLR [40,80,120,160] γ=0.5 | 200 | XRF55 paper |
| 03 | AdamW | 5e-4 | 1e-3 | warmup(10ep)+cosine → 4e-5 | 200 | APWMamba paper |

In [ ]:
# Cell 1 — Install mamba-ssm (required by WavMambaHAR BiMamba layers)
!pip install -q ninja packaging wheel
!pip install -q triton
!pip install -q causal-conv1d>=1.2.0 --no-build-isolation
!pip install -q mamba-ssm --no-build-isolation
print('Install done')

In [ ]:
# Cell 2 — Clone latest code from GitHub
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/WaveConvMamba')
if not CODE_PATH.exists():
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/imhoangt/WaveConvMamba.git',
         str(CODE_PATH)],
        check=True
    )
else:
    print('Repo already cloned.')

sys.path.insert(0, str(CODE_PATH))
print(f'Code path  : {CODE_PATH}')

from xrf55_bench.trainer import run
print('Import OK  : xrf55_bench.trainer.run')

In [ ]:
# Cell 3 — Configuration
from pathlib import Path
from xrf55_bench.config import TrainCfg, TrainCfg_for_protocol

SEEDS      = [42]               # single-seed default
# SEEDS    = [4, 8, 17, 42]    # multi-seed

# Data source — uncomment one:
BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/xrf55_amp_raw/raw')
# BENCH_DIR  = Path('/kaggle/input/datasets/imhoangt/xrf55-amp-dataset/xrf55_amp_processed/processed')

# Output dir — rename per run, e.g. wavmamba_p02_proc
OUTPUT_DIR = Path('/kaggle/working/outputs/wavmamba_p01_raw')

print(f'Seeds      : {SEEDS}')
print(f'Bench dir  : {BENCH_DIR}')
print(f'Output dir : {OUTPUT_DIR}')

for fname in ['stats.json', 'y_train.npy', 'y_test.npy',
              'wavmamba/X_train.npy', 'wavmamba/X_test.npy']:
    p = BENCH_DIR / fname
    status = 'OK' if p.exists() else 'MISSING'
    print(f'  [{status}] {p}')

In [ ]:
# Cell 4 — Run training
#
# Protocol presets:
#   01  AdamW  lr=1e-4  wd=0.01  no scheduler    40 ep   (tf_mamba paper)
#   02  Adam   lr=1e-3  wd=0.0   MultiStepLR     200 ep  (XRF55 paper)
#             milestones=[40,80,120,160]  gamma=0.5
#   03  AdamW  lr=5e-4  wd=1e-3  warmup+cosine   200 ep  (APWMamba paper)
#             warmup=10ep  floor_lr=4e-5  label_smoothing=0.1
#
# Uncomment one cfg line:
cfg = TrainCfg_for_protocol('01', seeds=tuple(SEEDS))            # 40ep,  AdamW lr=1e-4
# cfg = TrainCfg_for_protocol('02', seeds=tuple(SEEDS))          # 200ep, Adam  lr=1e-3, MultiStepLR
# cfg = TrainCfg_for_protocol('03', seeds=tuple(SEEDS))          # 200ep, AdamW lr=5e-4, warmup+cosine

run(model_name='wavmamba', bench_dir=BENCH_DIR, output_dir=OUTPUT_DIR,
    train_cfg=cfg, num_workers=4)

In [ ]:
# Cell 5 — Results
import json

metrics_path = OUTPUT_DIR / 'metrics.json'
if metrics_path.exists():
    with open(metrics_path) as f:
        m = json.load(f)
    s    = m['summary']
    seeds = m['config']['seeds']
    print(f"\n{'='*55}")
    print(f"  XRF55 Bench — WavMambaHAR")
    print(f"  Seeds       : {seeds}")
    print(f"  Accuracy    : {s['test_accuracy_mean']*100:.2f}% ± {s['test_accuracy_std']*100:.2f}%")
    print(f"  F1 Macro    : {s['test_f1_macro_mean']*100:.2f}% ± {s['test_f1_macro_std']*100:.2f}%")
    print(f"  Best epochs : {s['best_epochs']}")
    print(f"  Params      : {s['params_M']}M  |  Size: {s['model_size_mb']}MB")
    if s.get('macs_G'):
        print(f"  MACs        : {s['macs_G']:.3f}G")
    if s.get('latency_mean_ms') is not None:
        print(f"  Latency     : {s['latency_mean_ms']:.2f} ± {s['latency_std_ms']:.2f} ms")
    print(f"  Time        : {s['total_time_s']}s")
    print(f"{'='*55}")
    if len(seeds) == 1:
        ps = m['per_seed'].get(str(seeds[0]), {})
        if ps.get('test_f1_per_class'):
            class_names = [
                'Waving', 'Clap Hands', 'Fall on the Floor', 'Jumping', 'Running',
                'Sitting Down', 'Standing Up', 'Turning', 'Walking',
                'Stretch Oneself', 'Pat on Shoulder',
            ]
            print(f"\n  Per-class F1:")
            for i, v in enumerate(ps['test_f1_per_class']):
                print(f"    {class_names[i]:<20}: {v*100:.2f}%")
else:
    print('metrics.json not found — training may not have completed.')

In [ ]:
# Cell 6 — Plots + Download
import shutil
from IPython.display import Image, display, FileLink

for fname in ['training_curve.png', 'confusion_matrix.png', 'seed_comparison.png']:
    p = OUTPUT_DIR / 'plots' / fname
    if p.exists():
        display(Image(str(p)))

print('\n--- Download ---')
for fname in ['results_summary.zip', 'model.zip']:
    src = OUTPUT_DIR / fname
    dst = Path('/kaggle/working') / fname
    if src.exists():
        shutil.copy2(src, dst)
        size_mb = dst.stat().st_size / 1e6
        print(f'{fname}  ({size_mb:.1f} MB)')
        display(FileLink(fname))
    else:
        print(f'[MISSING] {fname} — run Cell 4 first.')